In [ ]:
# Install TensorFlow
%pip install tensorflow

In [ ]:
# Import pandas to load the CSV dataset
import pandas as pd

# Import NumPy for numerical operations
import numpy as np

# Import TensorFlow
import tensorflow as tf

# Import Keras from TensorFlow
from tensorflow import keras

# Import layers needed to build the CNN
from tensorflow.keras import layers

# Import train_test_split
from sklearn.model_selection import train_test_split

# Import matplotlib for visualization
import matplotlib.pyplot as plt

In [ ]:
# Display the installed TensorFlow version
print("TensorFlow version:", tf.__version__)

In [ ]:
# Load the MNIST training dataset
# Make sure train.csv is in the same folder as this notebook
train_data = pd.read_csv("../data/train.csv")

# Display the shape of the dataset
print("Training data shape:", train_data.shape)

In [ ]:
# X contains the pixel values
# Remove the label column because it is the target
X = train_data.drop("label", axis=1)

# y contains the actual digit labels from 0 to 9
y = train_data["label"]

# Display the shapes
print("Features shape:", X.shape)
print("Labels shape:", y.shape)

In [ ]:
# Convert the DataFrame into a NumPy array
# because TensorFlow works efficiently with NumPy arrays
X = X.values

# Convert pixel values from integers to floating-point values
X = X.astype("float32")

# Normalize pixel values from 0–255 to 0–1
X = X / 255.0

# Convert labels into a NumPy array
y = y.values

# Display the minimum and maximum pixel values
print("Minimum pixel value:", X.min())
print("Maximum pixel value:", X.max())

In [ ]:
# Reshape every image from 784 pixels
# into a 28 × 28 grayscale image
X = X.reshape(-1, 28, 28, 1)

# Display the new shape
print("Reshaped image data:", X.shape)

In [ ]:
# Split the images into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,                  # Image data
    y,                  # Digit labels
    test_size=0.2,      # 20% used for testing
    random_state=42,    # Makes the split reproducible
    stratify=y          # Keeps digit proportions balanced
)

# Display the resulting shapes
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

In [ ]:
# Create a figure for displaying 10 images
plt.figure(figsize=(10, 4))

# Display the first 10 training images
for i in range(10):

    # Create a subplot
    plt.subplot(2, 5, i + 1)

    # Display the image
    # [:, :, 0] removes the single grayscale channel
    plt.imshow(X_train[i, :, :, 0], cmap="gray")

    # Display the actual digit
    plt.title(f"Digit: {y_train[i]}")

    # Hide the axis
    plt.axis("off")

# Adjust spacing
plt.tight_layout()

# Display the images
plt.show()

In [ ]:
keras.utils.set_random_seed(42)

# Create a Sequential CNN model
model = keras.Sequential([

    # First convolutional layer
    # 32 filters of size 3 × 3
    # ReLU activation introduces non-linearity
    # input_shape represents a 28 × 28 grayscale image
    layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        input_shape=(28, 28, 1)
    ),

    # Reduce the spatial dimensions by taking
    # the maximum value from each 2 × 2 region
    layers.MaxPooling2D((2, 2)),

    # Second convolutional layer
    # This learns more complex image features
    layers.Conv2D(
        64,
        (3, 3),
        activation="relu"
    ),

    # Again reduce the image dimensions
    layers.MaxPooling2D((2, 2)),

    # Convert the 2D feature maps into a 1D vector
    # so they can be passed to the dense layers
    layers.Flatten(),

    # Fully connected layer with 128 neurons
    layers.Dense(
        128,
        activation="relu"
    ),

    # Output layer with 10 neurons
    # because there are 10 possible digits: 0–9
    layers.Dense(
        10,
        activation="softmax"
    )
])

In [ ]:
# Configure the CNN before training
model.compile(

    # Adam is the optimization algorithm
    optimizer="adam",

    # Sparse categorical crossentropy is suitable
    # because our labels are integers from 0 to 9
    loss="sparse_categorical_crossentropy",

    # Accuracy tells us how many images
    # are classified correctly
    metrics=["accuracy"]
)

In [ ]:
# Train the CNN using the training images
history = model.fit(

    # Training images
    X_train,

    # Training labels
    y_train,

    # Number of complete passes through the training data
    epochs=10,

    # Number of images processed at a time
    batch_size=128,

    # Reserve validation rows from the training split.
    validation_split=0.1,

    # Display training progress
    verbose=1
)

In [ ]:
# Plot the training accuracy
plt.figure(figsize=(8, 5))

# Plot accuracy values recorded during training
plt.plot(history.history["accuracy"], label="Training Accuracy")

# Plot validation accuracy
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

# Add the graph title
plt.title("CNN Training and Validation Accuracy")

# Label the x-axis
plt.xlabel("Epoch")

# Label the y-axis
plt.ylabel("Accuracy")

# Display the legend
plt.legend()

# Display the graph
plt.show()

In [ ]:
# Plot the training loss
plt.figure(figsize=(8, 5))

# Plot training loss values
plt.plot(history.history["loss"], label="Training Loss")

# Plot validation loss values
plt.plot(history.history["val_loss"], label="Validation Loss")

# Add the title
plt.title("CNN Training and Validation Loss")

# Label the x-axis
plt.xlabel("Epoch")

# Label the y-axis
plt.ylabel("Loss")

# Display the legend
plt.legend()

# Display the graph
plt.show()

In [ ]:
# Evaluate the CNN on the test dataset
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

# Display the test loss
print("CNN Test Loss:", test_loss)

# Display the test accuracy
print("CNN Test Accuracy:", test_accuracy)

# Display accuracy as a percentage
print("CNN Test Accuracy (%):", test_accuracy * 100)

In [ ]:
# Generate probability predictions for the test images
y_prob_cnn = model.predict(X_test)

# Convert the probability predictions
# into the class with the highest probability
y_pred_cnn = np.argmax(y_prob_cnn, axis=1)

# Display the first 10 predictions
print("First 10 CNN predictions:", y_pred_cnn[:10])

In [ ]:
# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Calculate CNN accuracy
cnn_accuracy = accuracy_score(
    y_test,
    y_pred_cnn
)

# Calculate weighted precision
cnn_precision = precision_score(
    y_test,
    y_pred_cnn,
    average="weighted"
)

# Calculate weighted recall
cnn_recall = recall_score(
    y_test,
    y_pred_cnn,
    average="weighted"
)

# Calculate weighted F1-score
cnn_f1 = f1_score(
    y_test,
    y_pred_cnn,
    average="weighted"
)

# Display the results
print("CNN Accuracy :", cnn_accuracy)
print("CNN Precision:", cnn_precision)
print("CNN Recall   :", cnn_recall)
print("CNN F1-Score :", cnn_f1)

In [ ]:
# Calculate the CNN error rate
cnn_error_rate = 1 - cnn_accuracy

# Count incorrect predictions
cnn_incorrect = (y_test != y_pred_cnn).sum()

# Count correct predictions
cnn_correct = (y_test == y_pred_cnn).sum()

# Display the results
print("CNN Error Rate:", cnn_error_rate)
print("CNN Error Rate (%):", cnn_error_rate * 100)

print("Correct predictions:", cnn_correct)
print("Incorrect predictions:", cnn_incorrect)

print("Total predictions:", len(y_test))

In [ ]:
# Import classification_report
from sklearn.metrics import classification_report

# Display precision, recall and F1-score
# for each digit from 0 to 9
print(classification_report(y_test, y_pred_cnn))

In [ ]:
# Import confusion matrix tools
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Calculate the confusion matrix
cm = confusion_matrix(
    y_test,
    y_pred_cnn
)

# Create the confusion matrix display
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=range(10)
)

# Display the matrix
disp.plot()

# Add a title
plt.title("CNN Confusion Matrix")

# Show the graph
plt.show()

In [ ]:
# Create a figure for displaying 10 test images
plt.figure(figsize=(10, 4))

# Display the first 10 test images
for i in range(10):

    # Create a subplot
    plt.subplot(2, 5, i + 1)

    # Display the image
    plt.imshow(
        X_test[i].reshape(28, 28),
        cmap="gray"
    )

    # Display actual and predicted labels
    plt.title(
        f"Actual: {y_test[i]}\n"
        f"Predicted: {y_pred_cnn[i]}"
    )

    # Hide the axis
    plt.axis("off")

# Adjust spacing
plt.tight_layout()

# Display the images
plt.show()

In [ ]:
# Save the trained CNN model
# The .keras format is recommended for modern Keras models
model.save("cnn_digit_model.keras")

# Confirm that the model was saved
print("CNN model saved successfully!")